# RAG Profile Matching: Experimentation and Analysis

This notebook explores the synthetic resume corpus, validates document processing, and provides optional indexing and job-matching experiments. Run it from the repository root with `uv run jupyter lab`.

> The optional indexing and matching cells call OpenAI and require `OPENAI_API_KEY`. The exploration cells do not send resume content to an external service.

In [1]:
from collections import Counter
import json
import os
import sys
from pathlib import Path
from time import perf_counter

ROOT = Path.cwd()
assert (ROOT / "dataset/manifest.json").is_file(), "Open the notebook from the repository root."
sys.path.insert(0, str(ROOT / "src"))

from filesSystemTools.fs_tools import list_files, read_file
from resume_rag import DB_DIR, extract_metadata, index_resumes, resume_paths, split_sections
from job_matcher import job_matches

manifest = json.loads((ROOT / "dataset/manifest.json").read_text())

## 1. Dataset overview

The manifest is the source of truth for the number of synthetic resumes and job descriptions.

In [2]:
resumes = manifest["resumes"]
jobs = manifest["job_descriptions"]
print(f"Resumes: {len(resumes)}")
print(f"Job descriptions: {len(jobs)}")
print(f"Unique roles: {len(Counter(resume['role'] for resume in resumes))}")
Counter(resume["role"] for resume in resumes).most_common()

Resumes: 35
Job descriptions: 6
Unique roles: 30


[('Data Scientist', 2),
 ('Backend Engineer', 2),
 ('Machine Learning Engineer', 2),
 ('Frontend Engineer', 2),
 ('Data Engineer', 2),
 ('Product Manager', 1),
 ('DevOps Engineer', 1),
 ('UX Researcher', 1),
 ('Cybersecurity Analyst', 1),
 ('Business Analyst', 1),
 ('Cloud Architect', 1),
 ('NLP Engineer', 1),
 ('Mobile Engineer', 1),
 ('QA Automation Engineer', 1),
 ('Financial Analyst', 1),
 ('Full Stack Engineer', 1),
 ('HR Operations Specialist', 1),
 ('Site Reliability Engineer', 1),
 ('Marketing Analyst', 1),
 ('Java Engineer', 1),
 ('AI Product Manager', 1),
 ('Platform Engineer', 1),
 ('Network Engineer', 1),
 ('Content Designer', 1),
 ('Operations Manager', 1),
 ('Data Analyst', 1),
 ('Security Engineer', 1),
 ('Solutions Architect', 1),
 ('QA Engineer', 1),
 ('Analytics Engineer', 1)]

## 2. File-system tools and section-aware parsing

`list_files` finds the documents and `read_file` parses the RTF resume. `split_sections` keeps the resume's meaningful sections separate for retrieval.

In [3]:
files = list_files(str(ROOT / "dataset/resumes"))
print(f"Files on disk: {len(files)}")
print(f"Unique resumes selected for indexing: {len(resume_paths(ROOT / 'dataset/resumes'))}")

sample_path = ROOT / "dataset/resumes/aisha_khan.rtf"
sample = read_file(str(sample_path))
assert sample["success"], sample
sections = split_sections(sample["content"])
sections

Files on disk: 70
Unique resumes selected for indexing: 35


{'Name': 'Aisha Khan',
 'Professional Summary': 'Data Scientist with 6 years of experience delivering measurable outcomes in collaborative teams.',
 'Skills': 'Python, SQL, Machine Learning, TensorFlow, AWS',
 'Experience': 'Data Scientist | 6 years Built demand forecasting models that improved inventory planning by 18%.',
 'Education': 'MSc Data Science'}

## 3. LLM metadata extraction

The indexer asks OpenAI for validated JSON containing `name`, `skills`, `experience_years`, and `education`. This cell is intentionally guarded because it sends the selected resume text to the configured OpenAI API.

In [4]:
if not os.environ.get("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY to run LLM metadata extraction.")
else:
    from openai import OpenAI
    extract_metadata(OpenAI(), sample["content"])

## 4. Optional indexing experiment

This creates or updates the local `chroma_db` collection. It makes one metadata-extraction request per resume and embedding requests for resume sections.

In [5]:
if not os.environ.get("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY before indexing.")
else:
    started = perf_counter()
    section_count = index_resumes()
    print(f"Indexed {section_count} sections in {perf_counter() - started:.2f} seconds.")

Indexed 140 sections in 44.44 seconds.


## 5. Job-matching experiment and latency

The matcher retrieves semantic candidates from ChromaDB, filters must-have skills and experience, and combines semantic similarity (70%) with skill coverage (30%).

In [6]:
job_description = "Data Engineer. Required Skills: Python, SQL, Airflow, 4+ years."

if not os.environ.get("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY before running a match.")
elif not (ROOT / DB_DIR).exists():
    print("Run the indexing cell first.")
else:
    started = perf_counter()
    match_result = job_matches(job_description)
    latency = perf_counter() - started
    print(f"Match latency: {latency:.2f} seconds")
    print(json.dumps(match_result, indent=2))

Match latency: 0.48 seconds
{
  "job_description": "Data Engineer. Required Skills: Python, SQL, Airflow, 4+ years.",
  "top_matches": [
    {
      "candidate_name": "Victoria Nguyen",
      "resume_path": "dataset/resumes/victoria_nguyen.rtf",
      "match_score": 90,
      "matched_skills": [
        "Python",
        "SQL",
        "Airflow"
      ],
      "relevant_excerpts": [
        "Skills\nPython, SQL, BigQuery, Airflow, Looker",
        "Professional Summary\nData Engineer with 4 years of experience delivering measurable outcomes in collaborative teams."
      ],
      "reasoning": "Matched skills: Python, SQL, Airflow; relevant sections: Education, Experience, Professional Summary, Skills."
    }
  ]
}


## 6. Analysis checklist

- **Retrieval relevance:** confirm that the top candidates have the required skills and relevant experience.
- **Must-have filtering:** change a requirement to `SQL or Airflow` and verify that candidates with either skill remain eligible.
- **Latency:** record the displayed match latency for several job descriptions.
- **Result quality:** inspect `matched_skills`, `relevant_excerpts`, and `reasoning` to confirm that scores have clear evidence.